---
title: "DRG Checks"

author: "Carlos Resurreccion"

date: "2025-04-03"
---


# Setup


## Parameters

Change which year_to_load to process in
`~/pids-drg-claims/data-cleaning/debug/cache/year_to_load.txt`

Change main GLOBAL (i.e. across all scripts) parameters in
`~/pids-drg-claims/data-cleaning/00a-parameters.r`

Change seldom touched parameters in
`~/pids-drg-claims/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R`


In [1]:
source(here::here("data-cleaning", "00a-parameters.r"))


Parallelization: TRUE 


## Libraries


In [2]:
source(here::here("data-cleaning", "00b-packages.r"))


Loading required package: data.table

Loading required package: here

here() starts at /home/resurreccion_cmc/pids-drg-claims

Loading required package: tictoc


Attaching package: ‘tictoc’


The following object is masked from ‘package:data.table’:

    shift


Loading required package: stringr

Loading required package: stringi

Loading required package: lubridate


Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


Loading required package: profvis

Loading required package: hash

hash-2.2.6.3 provided by Decision Patterns



Attaching package: ‘hash’


The following object is masked from ‘package:tictoc’:

    clear


The following object is masked from ‘package:data.table’:

    copy


Loading required package: future

Loading required package: future.apply

Load

## R Scripts


In [3]:
source(here::here("data-cleaning", "00c-load-params-and-scripts.r"))


==== Loaded Parameters ====

year_to_load: 2018

Automate: FALSE


Sourcing scripts from:/home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R

✅ Authentication successful using service account key.

All directories exist.


Total Rows via cached object: 11777674

Utilizing 4 cores (8 threads)


Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/0.2.0.process_helper_functions.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/0.3.0.grouping_functions.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/1.0.query_bq_to_dt.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/2.0.split_and_save_part.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/3.0.create_sample_files.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v

## Load Mapping Data


In [4]:
source(here::here("data-cleaning", "00d-load-mapping.r"))


==== Loading Data Mapping ====

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/proc.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/rvs_icd9.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/acr_rvs.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/tdrg_icd10.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/phl_icd10.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/i10vx.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/hci_2018.rds

00d-load-mapping.r successfully executed.



# Data Verification Proper


## File Loading


### Load final .rds


In [5]:
dt_clean <- readRDS(here(
  chkpt_2_path,
  paste0(
    chkpt_2_prefix, year_to_load, suffix,
    "v2", "_part_b_bq_subset", ".rds"
  )
))


### Load raw file


In [6]:
dt_raw <- fread(
  file = here(
    raw_claims_path,
    paste0(full_claims_prefix, year_to_load, file_type)
  ), colClasses = "character", header = TRUE,
  encoding = "Latin-1", na.strings = na_values
)

# Drop columns
cols_to_drop <- intersect(colnames(dt_raw), c(drop_cols, drop_cols_manual))
if (length(cols_to_drop) > 0) {
  dt_raw <- dt_raw[, (cols_to_drop) := NULL]
}

avail_cols <- colnames(dt_raw)

# Rename Columns
setnames(dt_raw,
  old = avail_cols[avail_cols %in% names(column_mappings)],
  new = unlist(column_mappings[
    avail_cols[
      avail_cols %in% names(column_mappings)
    ]
  ])
)

dt_raw[dt_raw == ""] <- NA_character_
dt_raw[, id_series := trimws(id_series)]


Warning message in fread(file = here(raw_claims_path, paste0(full_claims_prefix, :
“na.strings[16]==" " consists only of whitespace, ignoring. strip.white==TRUE (default) and "" is present in na.strings, so any number of spaces in string columns will already be read as <NA>.”


## Definitions


### Function Definitions


In [7]:
test_checks <- function(section_id) {
  # Coerce to two-character string (e.g., 1 → "01")
  section_id <- sprintf("%02d", as.integer(section_id))

  # Get the calling environment (e.g., global or wherever this is invoked from)
  calling_env <- parent.frame()

  # Build pattern to match only variables for the given section
  pattern <- paste0("^chk_", section_id, "_\\d{2}_.+")

  # List relevant check variables in the calling environment
  chk_vars <- ls(envir = calling_env, pattern = pattern)

  # If no matching checks found, warn and exit
  if (length(chk_vars) == 0) {
    cat("⚠️ No checks found for section: ", section_id)
    return(invisible(NULL))
  }

  # Get values (assumed format: c(flag, info))
  chk_values_raw <- lapply(chk_vars, get, envir = calling_env)

  # Extract just the logical flag from each
  chk_flags <- sapply(chk_values_raw, function(x) isTRUE(x[1]))

  # Check for failures
  if (any(!chk_flags)) {
    failed_checks <- chk_vars[!chk_flags]

    # Build detailed failure messages
    failure_messages <- mapply(function(var, val) {
      suffix <- sub(paste0("^chk_", section_id, "_\\d{2}_(.+)$"), "\\1", var)
      info <- if (length(val) > 1) val[2] else "No additional info"
      paste0("• ", suffix, ": ", info)
    }, var = failed_checks, val = chk_values_raw[!chk_flags], SIMPLIFY = TRUE)

    stop(paste0(
      "❌ Validation failed in section ", section_id, ":\n",
      paste(failure_messages, collapse = "\n")
    ))
  } else {
    # Print all check results
    cat(paste0(
      "✅ All validation checks passed for section ",
      section_id, ":\n"
    ))
    for (i in seq_along(chk_vars)) {
      suffix <- sub(paste0(
        "^chk_", section_id,
        "_\\d{2}_(.+)$"
      ), "\\1", chk_vars[i])
      cat(paste0(suffix, ": ", chk_values_raw[[i]][1], "\n"))
    }
  }
}

custom_setdiff <- function(x, y) {
  # If equal, return TRUE with no message
  if (setequal(x, y)) {
    return(c(TRUE, NULL))
  } else {
    # Identify elements missing and extra
    missing_in_y <- setdiff(x, y) # present in x but missing in y
    extra_in_y <- setdiff(y, x) # present in y but not in x

    # Compose detailed message
    mismatch_msg <- paste0(
      if (length(missing_in_y)) {
        paste0("\n  - missing: ", paste(missing_in_y, collapse = ", "))
      } else {
        ""
      },
      if (length(extra_in_y)) {
        paste0("\n  - invalid: ", paste(extra_in_y, collapse = ", "))
      } else {
        ""
      }
    )

    return(c(FALSE, mismatch_msg))
  }
}

custom_setequal <- function(x, y) {
  # If equal, return TRUE with no message
  if (setequal(x, y)) {
    return(c(TRUE, NULL))
  } else {
    mismatch_msg <- paste0(
      "X: ", x, " Y: ", y
    )
    return(c(FALSE, mismatch_msg))
  }
}

custom_check_mappings <- function(result_list) {
  # Keep only the FALSE mappings
  filtered <- lapply(result_list, function(x) Filter(Negate(isTRUE), x))
  filtered <- Filter(length, filtered) # drop empty lists

  if (length(filtered) == 0) {
    return(c(TRUE, NULL))
  } else {
    ticker <- 0
    for (col in names(filtered)) {
      msg_lines <- if (ticker == 0) {
        c(paste0("\n $ ", col, ":"))
      } else {
        c(msg_lines, paste0(" $ ", col, ":"))
      }
      ticker <- ticker + 1
      for (raw_val in names(filtered[[col]])) {
        # Extract the actual wrongly mapped value from the original result_list
        wrong_val <- result_list[[col]][[raw_val]]
        msg_lines <- c(
          msg_lines,
          paste0('   $ "', raw_val, '": MAPPING FAILED')
        )
      }
    }
    return(c(FALSE, paste(msg_lines, collapse = "\n")))
  }
}


## Checks Definition


### Test Batch 01:


In [8]:
cols_expected <- bq_cols
cols_actual <- colnames(dt_clean)
cols_schema <- fromJSON(here(
  "data-cleaning/r_scripts_v2",
  "bq_schema_cleaning.json"
))$name
chk_01_01_cols_match_expected <- custom_setdiff(cols_expected, cols_actual)
chk_01_02_cols_match_schema <- custom_setdiff(cols_schema, cols_actual)
test_checks(1)


✅ All validation checks passed for section 01:
cols_match_expected: TRUE
cols_match_schema: TRUE


### Test Batch 02:


In [9]:
nrow_expected <- dt_raw[, .N]
nrow_actual_full <- nrow(dt_clean)
chk_02_01_nrows_match_full <- custom_setequal(
  nrow_expected, nrow_actual_full
)

# Function to check if MD5 hashes have changed, returning TRUE if no change
check_md5_changes <- function(year_to_load) {
  # Function to calculate and save MD5 hash for a given file
  calculate_md5 <- function(file_path) {
    md5sum <- digest::digest(file_path, algo = "md5", file = TRUE)
    return(md5sum)
  }
  hash_cache_dir <- here::here("data-cleaning/debug/cache/partial_md5")
  dir.create(hash_cache_dir, recursive = TRUE, showWarnings = FALSE)
  hash_file_path <- here::here(
    hash_cache_dir,
    paste0("md5_hashes_", year_to_load, ".rds")
  )

  # Generate new MD5 hashes for each part
  current_hashes <- sapply(1:split_parts, function(part) {
    part_file <- here::here(
      raw_claims_parts_path,
      paste0(
        full_claims_prefix, year_to_load, "_part_",
        sprintf("%02d", part), "_of_", split_parts, ".rds"
      )
    )
    calculate_md5(part_file)
  })

  # Check if saved hashes exist
  if (file.exists(hash_file_path)) {
    saved_hashes <- readRDS(hash_file_path)
    # Return TRUE if hashes match, indicating no changes
    if (identical(saved_hashes, current_hashes)) {
      cat(paste(
        "✅ No changes in partial files for eclaims year",
        year_to_load, "\n"
      ))
      return(TRUE)
    }
  }
  return(FALSE)
}
nrow_actual_partial <- nrow_partial <- 0
if (!check_md5_changes(year_to_load)) {
  for (loop_part in 1:split_parts) {
    nrow_partial <- nrow(read_appropriate_file(loop_part))
    nrow_actual_partial <- nrow_actual_partial + nrow_partial
  }
  chk_02_02_nrows_match_partial <- custom_setequal(
    nrow_expected, nrow_actual_partial
  )
} else {
  chk_02_02_nrows_match_partial <- c(TRUE, NULL)
}

test_checks(2)


✅ No changes in partial files for eclaims year 2018 
✅ All validation checks passed for section 02:
nrows_match_full: TRUE
nrows_match_partial: TRUE


### Test Batch 03:


In [10]:
id_series_rows <- dt_clean[grepl("e", id_series), .(id_series)]
id_pin_rows <- dt_clean[grepl("e", id_pin), .(id_series, id_pin)]
id_hci_rows <- dt_clean[grepl("e", id_hci), .(id_series, id_hci)]
id_series_expo <-
  if (!is.null(id_series_rows)) {
    nrow(id_series_rows)
  } else {
    0
  }
id_pin_expo <-
  if (!is.null(id_pin_rows)) {
    nrow(id_pin_rows)
  } else {
    0
  }
id_hci_expo <-
  if (!is.null(id_hci_rows)) {
    nrow(id_hci_rows)
  } else {
    0
  }
chk_03_01_id_series_expo <-
  custom_setequal(id_series_expo, 0)
chk_03_02_id_pin_expo <-
  custom_setequal(id_pin_expo, 0)
chk_03_03_id_hci_expo <-
  custom_setequal(id_hci_expo, 0)
test_checks(3)


✅ All validation checks passed for section 03:
id_series_expo: TRUE
id_pin_expo: TRUE
id_hci_expo: TRUE


### Test Batch 04:


In [ ]:
mapping_results <- list()
if (to_debug) {
  cat("==================================================\n")
  flush.console()
}
# Subfunction: check mapping for one column
process_column_mapping <- function(
    col_name, dt_raw, dt_clean,
    expected_mappings, to_debug = FALSE) {
  if (to_debug) {
    cat(col_name, "\n--------------------------------------------------\n")
    flush.console()
  }
  result <- list()
  raw_col <- dt_raw[[col_name]]
  clean_col <- dt_clean[[col_name]]
  unique_raw_vals <- unique(na.omit(raw_col))
  for (raw_val in unique_raw_vals) {
    if (to_debug) {
      cat(raw_val, "\n")
      flush.console()
    }
    id_subset <- dt_raw[raw_col == raw_val, id_series]
    clean_subset <- unlist(
      dt_clean[id_series %chin% id_subset, .SD, .SDcols = col_name],
      use.names = FALSE
    )
    unique_clean_vals <- unique(na.omit(clean_subset))
    result_flag <-
      all(is.na(unique_clean_vals)) | length(unique_clean_vals) == 1
    result[[raw_val]] <- result_flag
    if (to_debug) {
      cat(str(id_subset), str(clean_subset), str(unique_clean_vals), sep = "\n")
      cat(result_flag, "\n--------------------------------------------------\n")
      flush.console()
    }
  }
  return(result)
}

mapping_results <- list()
if (to_debug) cat("==================================================\n")
if (to_debug) flush.console()
for (col_name in names(expected_mappings)) {
  mapping_results[[col_name]] <-
    process_column_mapping(
      col_name, dt_raw, dt_clean,
      expected_mappings, to_debug
    )
  if (to_debug) cat("==================================================\n")
  if (to_debug) flush.console()
}

if (to_debug) {
  str(mapping_results)
}

chk_04_01_mapping_results <- custom_check_mappings(mapping_results)
test_checks(04)


✅ All validation checks passed for section 04:
mapping_results: TRUE
